# ROGII — Wellbore Geology Prediction
## Advanced Professional Baseline

**Author:** Md Ashraf  
**Institute:** IIT (ISM) Dhanbad  
**Metric:** RMSE  
**Target:** TVT (True Vertical Thickness)  

---

### Pipeline Overview
1. Load train / test horizontal well data  
2. Construct submission-compatible `id` for test rows  
3. Feature engineering (rolling, lag, gradient, trajectory)  
4. GroupKFold training — grouped by WELL to prevent leakage  
5. OOF validation + test inference  
6. Correct id-based submission merge  

## 1. Imports

In [4]:
import os
import gc
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

import matplotlib.pyplot as plt
import plotly.express as px
import seaborn as sns

from sklearn.model_selection import GroupKFold
from sklearn.metrics import mean_squared_error

import lightgbm as lgb

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 200)

print(f"pandas  : {pd.__version__}")
print(f"numpy   : {np.__version__}")
print(f"lightgbm: {lgb.__version__}")

pandas  : 2.2.1
numpy   : 1.26.4
lightgbm: 4.6.0


## 2. Paths & Configuration

In [8]:
# ── Adjust ROOT_DIR to match your environment ──────────────────────────────
# LOCAL
ROOT_DIR = Path("C:\\Users\\MD ASHRAF\\Documents\\wellbore-geology-prediction-Well-log\\data\\raw")
# KAGGLE (uncomment when submitting)
# ROOT_DIR = Path("/kaggle/input/rogii-wellbore-geology-prediction")

TRAIN_DIR       = ROOT_DIR / "train"
TEST_DIR        = ROOT_DIR / "test"
SUBMISSION_PATH = ROOT_DIR / "sample_submission.csv"

OUTPUT_DIR = Path("./outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

# ── Columns that are train-only and must NEVER appear as features ──────────
TRAIN_ONLY_COLS = ["ANCC", "ASTNU", "ASTNL", "EGFDU", "EGFDL", "BUDA"]

# ── Columns to always exclude from the feature matrix ─────────────────────
EXCLUDE_COLS = ["WELL", "TVT", "id"] + TRAIN_ONLY_COLS

N_FOLDS    = 5
RANDOM_SEED = 42

print("Paths OK")
print(f"  TRAIN : {TRAIN_DIR}")
print(f"  TEST  : {TEST_DIR}")

Paths OK
  TRAIN : C:\Users\MD ASHRAF\Documents\wellbore-geology-prediction-Well-log\data\raw\train
  TEST  : C:\Users\MD ASHRAF\Documents\wellbore-geology-prediction-Well-log\data\raw\test


## 3. Metric

In [9]:
def rmse(y_true, y_pred):
    return np.sqrt(mean_squared_error(y_true, y_pred))

## 4. Load Data

In [10]:
def load_horizontal_wells(directory: Path, split: str) -> pd.DataFrame:
    """
    Load all *__horizontal_well.csv files from a directory.

    Adds:
        WELL     — well name extracted from filename
        ROW_IDX  — per-well integer index (0-based)
        id       — submission-compatible key: "{WELL}_{ROW_IDX}"
    """
    files = sorted(directory.glob("*__horizontal_well.csv"))
    if not files:
        raise FileNotFoundError(f"No horizontal well files found in {directory}")

    frames = []
    for f in tqdm(files, desc=f"Loading {split}"):
        well_name = f.stem.split("__")[0]
        df = pd.read_csv(f)
        df["WELL"]    = well_name
        df["ROW_IDX"] = np.arange(len(df))
        # FIX: build the id key that matches sample_submission.csv
        df["id"]      = well_name + "_" + df["ROW_IDX"].astype(str)
        frames.append(df)

    out = pd.concat(frames, ignore_index=True)
    print(f"{split} shape : {out.shape}")
    print(f"{split} wells : {out['WELL'].nunique()}")
    return out


train_df = load_horizontal_wells(TRAIN_DIR, "TRAIN")
test_df  = load_horizontal_wells(TEST_DIR,  "TEST")

Loading TRAIN:   0%|          | 0/773 [00:00<?, ?it/s]

TRAIN shape : (5092255, 16)
TRAIN wells : 773


Loading TEST:   0%|          | 0/3 [00:00<?, ?it/s]

TEST shape : (19221, 9)
TEST wells : 3


## 5. Quick Sanity Checks

In [11]:
print("── Train columns ──")
print(list(train_df.columns))

print("\n── Test columns ──")
print(list(test_df.columns))

print("\n── TVT statistics ──")
print(train_df["TVT"].describe())

print(f"\nTVT null rows in train : {train_df['TVT'].isna().sum()}")
print(f"TVT_input null rows in train : {train_df['TVT_input'].isna().sum()}")
print(f"TVT_input null rows in test  : {test_df['TVT_input'].isna().sum()}")

── Train columns ──
['MD', 'X', 'Y', 'Z', 'ANCC', 'ASTNU', 'ASTNL', 'EGFDU', 'EGFDL', 'BUDA', 'TVT', 'GR', 'TVT_input', 'WELL', 'ROW_IDX', 'id']

── Test columns ──
['MD', 'X', 'Y', 'Z', 'GR', 'TVT_input', 'WELL', 'ROW_IDX', 'id']

── TVT statistics ──
count    5.092255e+06
mean     1.150364e+04
std      6.399711e+02
min      9.245190e+03
25%      1.098793e+04
50%      1.135451e+04
75%      1.203826e+04
max      1.289389e+04
Name: TVT, dtype: float64

TVT null rows in train : 0
TVT_input null rows in train : 3783989
TVT_input null rows in test  : 14151


## 6. Verify Sample Submission ID Format

In [12]:
# ── FIX: validate id construction BEFORE training ─────────────────────────
sample_sub = pd.read_csv(SUBMISSION_PATH)
print(f"Sample submission shape : {sample_sub.shape}")
print(sample_sub.head())

# Check how many test ids appear in the submission
sub_ids  = set(sample_sub["id"])
test_ids = set(test_df["id"])

matched   = sub_ids & test_ids
unmatched = sub_ids - test_ids

print(f"\nSubmission rows    : {len(sub_ids)}")
print(f"Test rows built    : {len(test_ids)}")
print(f"Matched            : {len(matched)}")
print(f"Unmatched (in sub but NOT in test) : {len(unmatched)}")

if unmatched:
    print("\n⚠  ID mismatch detected. Review id construction above.")
    print("   Sample unmatched ids :", list(unmatched)[:5])
    print("   Sample test ids      :", list(test_ids)[:5])
else:
    print("\n✓  All submission IDs match test data.")

Sample submission shape : (14151, 2)
              id  tvt
0  000d7d20_1442  0.0
1  000d7d20_1443  0.0
2  000d7d20_1444  0.0
3  000d7d20_1445  0.0
4  000d7d20_1446  0.0

Submission rows    : 14151
Test rows built    : 19221
Matched            : 14151
Unmatched (in sub but NOT in test) : 0

✓  All submission IDs match test data.


## 7. Exploratory Data Analysis

In [ ]:
# Target distribution
fig = px.histogram(
    train_df, x="TVT", nbins=100,
    title="TVT Distribution — Training Set",
    template="plotly_white"
)
fig.show()

In [ ]:
# Sample well log view
sample_well = train_df["WELL"].unique()[0]
well_df = train_df[train_df["WELL"] == sample_well]

fig = px.line(
    well_df, x="MD", y=["GR", "TVT_input", "TVT"],
    title=f"Sample Well: {sample_well} — GR, TVT_input, TVT vs MD",
    template="plotly_white"
)
fig.show()

In [ ]:
# Wells per fold preview
print("Rows per well (top 10):")
print(train_df.groupby("WELL").size().sort_values(ascending=False).head(10))

## 8. Feature Engineering

In [ ]:
def build_features(df: pd.DataFrame) -> pd.DataFrame:
    """
    All feature engineering in one place.
    Operates on a copy; grouped by WELL to prevent cross-well leakage.
    """
    df = df.copy()

    # ── Rolling windows ────────────────────────────────────────────────────
    windows = [5, 10, 20, 50]

    for w in windows:
        grp_gr  = df.groupby("WELL")["GR"]
        grp_tvt = df.groupby("WELL")["TVT_input"]

        df[f"GR_roll_mean_{w}"]      = grp_gr.transform(lambda x: x.rolling(w, min_periods=1).mean())
        df[f"GR_roll_std_{w}"]       = grp_gr.transform(lambda x: x.rolling(w, min_periods=1).std())
        df[f"TVT_input_roll_mean_{w}"] = grp_tvt.transform(lambda x: x.rolling(w, min_periods=1).mean())
        df[f"TVT_input_roll_std_{w}"]  = grp_tvt.transform(lambda x: x.rolling(w, min_periods=1).std())

    # ── Lag features ───────────────────────────────────────────────────────
    lags = [1, 2, 5, 10]

    for lag in lags:
        df[f"GR_lag_{lag}"]        = df.groupby("WELL")["GR"].shift(lag)
        df[f"TVT_input_lag_{lag}"] = df.groupby("WELL")["TVT_input"].shift(lag)

    # ── Gradient (diff) features ───────────────────────────────────────────
    df["GR_gradient"]        = df.groupby("WELL")["GR"].diff()
    df["TVT_input_gradient"] = df.groupby("WELL")["TVT_input"].diff()

    # 2nd derivative — rate of change of gradient
    df["GR_gradient2"]        = df.groupby("WELL")["GR_gradient"].diff()
    df["TVT_input_gradient2"] = df.groupby("WELL")["TVT_input_gradient"].diff()

    # ── Trajectory features ────────────────────────────────────────────────
    df["dX"] = df.groupby("WELL")["X"].diff()
    df["dY"] = df.groupby("WELL")["Y"].diff()
    df["dZ"] = df.groupby("WELL")["Z"].diff()

    df["traj_distance"] = np.sqrt(
        df["dX"]**2 + df["dY"]**2 + df["dZ"]**2
    )

    # Inclination proxy — vertical drop per unit lateral move
    lateral = np.sqrt(df["dX"]**2 + df["dY"]**2).replace(0, np.nan)
    df["inclination_proxy"] = df["dZ"].abs() / lateral

    # ── TVT_input last-known forward fill ──────────────────────────────────
    # This is valid: we only use values already observed up to current MD
    df["TVT_input_ffill"] = df.groupby("WELL")["TVT_input"].transform(
        lambda x: x.ffill()
    )

    # ── GR relative to rolling mean (de-trended) ───────────────────────────
    df["GR_detrend_10"] = df["GR"] - df["GR_roll_mean_10"]
    df["GR_detrend_50"] = df["GR"] - df["GR_roll_mean_50"]

    # ── FIX: fill NaNs from lags/diffs using forward then backward fill ────
    # Per-well to avoid cross-well bleed
    df = df.groupby("WELL", group_keys=False).apply(
        lambda g: g.ffill().bfill()
    )

    return df


print("Building features for TRAIN...")
train_df = build_features(train_df)

print("Building features for TEST...")
test_df  = build_features(test_df)

print(f"\nTrain shape after features : {train_df.shape}")
print(f"Test  shape after features : {test_df.shape}")

## 9. Build Feature Matrix

In [ ]:
# ── FIX: derive features from intersection of train & test columns,
#         explicitly removing all non-feature columns ─────────────────────

common_cols = set(train_df.columns) & set(test_df.columns)

features = sorted(
    col for col in common_cols
    if col not in EXCLUDE_COLS
    and col != "ROW_IDX"     # index, not a feature
    and not col.startswith("_")
)

print(f"Total features: {len(features)}")
print(features)

# Double-check: no train-only surface columns snuck in
leaked = [c for c in TRAIN_ONLY_COLS if c in features]
assert not leaked, f"Surface top columns leaked into features: {leaked}"

In [ ]:
# Only rows where TVT is labelled (train target is present)
train_mask = train_df["TVT"].notna()

X      = train_df.loc[train_mask, features].reset_index(drop=True)
y      = train_df.loc[train_mask, "TVT"].reset_index(drop=True)
groups = train_df.loc[train_mask, "WELL"].reset_index(drop=True)

X_test = test_df[features].reset_index(drop=True)

print(f"X      : {X.shape}")
print(f"y      : {y.shape}")
print(f"X_test : {X_test.shape}")
print(f"NaN in X      : {X.isna().sum().sum()}")
print(f"NaN in X_test : {X_test.isna().sum().sum()}")

## 10. LightGBM — GroupKFold Training

In [ ]:
params = {
    "objective"       : "regression",
    "metric"          : "rmse",
    "boosting_type"   : "gbdt",
    "learning_rate"   : 0.03,
    "num_leaves"      : 64,
    "feature_fraction": 0.8,
    "bagging_fraction": 0.8,
    "bagging_freq"    : 5,
    "min_child_samples": 20,
    "n_estimators"    : 5000,
    "random_state"    : RANDOM_SEED,
    "verbosity"       : -1,
    "n_jobs"          : -1,
}

folds = GroupKFold(n_splits=N_FOLDS)

oof_preds  = np.zeros(len(X))
test_preds = np.zeros(len(X_test))
fold_scores = []
fold_models = []

print(f"Starting {N_FOLDS}-fold GroupKFold training...\n")

for fold, (tr_idx, val_idx) in enumerate(
    folds.split(X, y, groups), start=1
):
    print("-" * 55)
    print(f"  FOLD {fold}/{N_FOLDS}")
    print(f"  Train wells : {groups.iloc[tr_idx].nunique()}")
    print(f"  Valid wells : {groups.iloc[val_idx].nunique()}")
    print("-" * 55)

    X_tr, y_tr = X.iloc[tr_idx], y.iloc[tr_idx]
    X_val, y_val = X.iloc[val_idx], y.iloc[val_idx]

    model = lgb.LGBMRegressor(**params)
    model.fit(
        X_tr, y_tr,
        eval_set=[(X_val, y_val)],
        callbacks=[
            lgb.early_stopping(stopping_rounds=200, verbose=False),
            lgb.log_evaluation(period=500)
        ]
    )

    val_preds = model.predict(X_val)
    oof_preds[val_idx] = val_preds

    score = rmse(y_val, val_preds)
    fold_scores.append(score)
    fold_models.append(model)

    # Accumulate test predictions averaged across folds
    test_preds += model.predict(X_test) / N_FOLDS

    print(f"  → Fold RMSE : {score:.5f}")

    gc.collect()

print("\n" + "=" * 55)
print(f"  OOF RMSE : {rmse(y, oof_preds):.5f}")
print(f"  CV mean  : {np.mean(fold_scores):.5f}  ±  {np.std(fold_scores):.5f}")
print("=" * 55)

## 11. Feature Importance

In [ ]:
# Average importance across all folds
importance_matrix = np.column_stack(
    [m.feature_importances_ for m in fold_models]
)
mean_importance = importance_matrix.mean(axis=1)

importance_df = (
    pd.DataFrame({"feature": features, "importance": mean_importance})
    .sort_values("importance", ascending=False)
    .reset_index(drop=True)
)

print("Top 20 features:")
print(importance_df.head(20).to_string(index=False))

fig = px.bar(
    importance_df.head(30),
    x="importance", y="feature",
    orientation="h",
    title="Top 30 Features — Mean Importance Across Folds",
    template="plotly_white"
)
fig.update_layout(yaxis={"categoryorder": "total ascending"})
fig.show()

## 12. OOF Analysis

In [ ]:
oof_df = pd.DataFrame({
    "WELL"    : groups,
    "TVT_true": y.values,
    "TVT_pred": oof_preds,
})
oof_df["residual"] = oof_df["TVT_true"] - oof_df["TVT_pred"]

# Per-well RMSE
well_rmse = (
    oof_df.groupby("WELL")
    .apply(lambda g: rmse(g["TVT_true"], g["TVT_pred"]))
    .rename("RMSE")
    .sort_values(ascending=False)
    .reset_index()
)

print("Worst 10 wells by RMSE:")
print(well_rmse.head(10).to_string(index=False))

fig = px.histogram(
    oof_df, x="residual", nbins=100,
    title="OOF Residual Distribution",
    template="plotly_white"
)
fig.show()

## 13. Create Submission

> **FIX:** Predictions are merged on `id` — not sliced by position — so row order never matters.

In [ ]:
# Build prediction dataframe keyed on id
pred_df = test_df[["id"]].copy().reset_index(drop=True)
pred_df["tvt"] = test_preds

# ── FIX: merge on id so submission order is always correct ────────────────
submission = sample_sub[["id"]].merge(pred_df, on="id", how="left")

# Sanity: no NaN predictions
n_missing = submission["tvt"].isna().sum()
if n_missing:
    print(f"⚠  {n_missing} rows have NaN predictions — filling with global median")
    submission["tvt"] = submission["tvt"].fillna(y.median())
else:
    print("✓  All submission rows have valid predictions")

assert len(submission) == len(sample_sub), "Submission row count mismatch!"

out_path = OUTPUT_DIR / "submission.csv"
submission.to_csv(out_path, index=False)

print(f"\nSubmission saved → {out_path}")
print(f"Shape : {submission.shape}")
print(submission.head(10))

## 14. Summary

In [ ]:
print("="*55)
print(" ROGII BASELINE — FINAL SUMMARY")
print("="*55)
print(f"  Features          : {len(features)}")
print(f"  Train rows used   : {len(X)}")
print(f"  Test rows         : {len(X_test)}")
print(f"  Folds             : {N_FOLDS} (GroupKFold by WELL)")
for i, s in enumerate(fold_scores, 1):
    print(f"  Fold {i} RMSE       : {s:.5f}")
print(f"  OOF RMSE (final)  : {rmse(y, oof_preds):.5f}")
print(f"  Submission        : {out_path}")
print("="*55)
print("\nPIPELINE COMPLETED SUCCESSFULLY")